In [1]:
%pip install -q google-genai python-dotenv

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
from dotenv import load_dotenv

load_dotenv()

GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")

if GOOGLE_API_KEY:
    print("API Key loaded successfully!")
else:
    print("API Key not found!")

API Key loaded successfully!


In [3]:
from google import genai

client = genai.Client(api_key=GOOGLE_API_KEY)

print("Gemini client initialized successfully!")

Gemini client initialized successfully!


In [6]:
for model in client.models.list():
    print(model.name)

models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemma-4-26b-a4b-it
models/gemma-4-31b-it
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3.1-flash-lite
models/gemini-3-pro-image
models/gemini-3.1-flash-image
models/gemini-3.1-flash-lite-image
models/gemini-nano-banana-2.1
models/gemini-3.5-flash
models/gemini-3.5-flash-lite
models/gemini-omni-flash-preview
models/gemini-3.6-flash
models/gemini-3.7-flash
models/gemini-3.8-flash
models/gemini-4-argon
models/gemini-4-carbon
models/gemini-4-hydrogen
models/antigravity-preview-05-2026
models/antigravity-preview-09-2026
models/antigravity-preview-latest
models/deep-research-max-preview-04-2026
models/deep-research-preview-04-2026
models/deep-research-pro-preview-12-2025
models/gemini-embedding-001
models/gemini-embedding-2


In [7]:
interaction = client.interactions.create(
    model="gemini-4-hydrogen",
    input="Explain how AI works in a few words"
)
print(interaction.output_text)

AI learns patterns from massive amounts of data to make predictions and decisions.


In [8]:
%pip install -U langchain langchain-google-genai python-dotenv pydantic

  Using cached httpx-0.28.1-py3-none-any.whl.metadata (7.1 kB)
  Using cached langchain_protocol-0.0.21-py3-none-any.whl.metadata (2.4 kB)
  Using cached langsmith-0.14.17-py3-none-any.whl.metadata (39 kB)
  Using cached pyyaml-6.0.3-cp314-cp314-win_amd64.whl.metadata (2.4 kB)
  Using cached uuid_utils-1.2.0-cp310-abi3-win_amd64.whl.metadata (6.5 kB)
  Using cached langgraph_sdk-0.4.14-py3-none-any.whl.metadata (5.1 kB)
  Using cached xxhash-4.1.0-cp314-cp314-win_amd64.whl.metadata (19 kB)
  Using cached ormsgpack-1.13.1-cp314-cp314-win_amd64.whl.metadata (3.3 kB)
  Using cached orjson-3.13.0-cp314-cp314-win_amd64.whl.metadata (43 kB)
  Using cached requests_toolbelt-1.0.0-py2.py3-none-any.whl.metadata (14 kB)
  Using cached uuid_utils-0.19.0-cp310-abi3-win_amd64.whl.metadata (6.5 kB)
  Using cached zstandard-0.25.0-cp314-cp314-win_amd64.whl.metadata (3.3 kB)
  Using cached filetype-1.2.0-py2.py3-none-any.whl.metadata (6.5 kB)
  Using cached httpcore-1.0.9-py3-none-any.whl.metadata (21

In [9]:
import os
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI

load_dotenv()

llm = ChatGoogleGenerativeAI(
    model="gemini-4-hydrogen",
    temperature=0,
    google_api_key=os.getenv("GOOGLE_API_KEY")
)

print("Gemini connected through LangChain!")

Gemini connected through LangChain!


In [10]:
response = llm.invoke(
    "Explain the purpose of an AI-powered GitHub PR reviewer."
)

print(response.content)

[{'type': 'text', 'text': 'An **AI-powered GitHub Pull Request (PR) reviewer** is an automated tool integrated into a repository\'s workflow that analyzes code changes and provides feedback before—or alongside—human reviewers. \n\nIts primary purpose is to accelerate the software development lifecycle and improve code quality by acting as an always-available "first line of defense" in code reviews.\n\nHere are the main purposes and benefits it serves:\n\n### 1. Eliminating Review Bottlenecks\nHuman reviewers often take hours or days to get to a PR due to meetings, timezone differences, or deep work. An AI reviewer runs immediately when a PR is opened or updated, giving the author instant feedback so they can fix obvious issues before a teammate even looks at the code.\n\n### 2. Catching Bugs and Security Vulnerabilities Early\nAI reviewers scan diffs for common logic errors, edge cases (like unhandled null values or race conditions), performance bottlenecks, and security risks (such as

In [11]:
from agents.code_reviewer import review_code_diff

filename = "auth.js"

diff = """
+app.get("/user", (req, res) => {
+    const userId = req.query.id;
+    const query = "SELECT * FROM users WHERE id = " + userId;
+    db.execute(query, (err, result) => {
+        res.json(result);
+    });
+});
"""

result = review_code_diff(filename, diff)

print(result)

{'summary': 'The new /user endpoint in auth.js contains a critical SQL injection vulnerability, ignores database errors, and potentially exposes sensitive user columns.', 'findings': [{'serverity': 'critical', 'category': 'security', 'file': 'auth.js', 'line': None, 'title': 'SQL injection via unsanitized query parameter', 'description': "Concatenating req.query.id directly into the SQL string allows attackers to execute arbitrary SQL. Use a parameterized query instead, such as db.execute('SELECT id, username, email FROM users WHERE id = ?', [userId], ...)."}, {'serverity': 'High', 'category': 'bug', 'file': 'auth.js', 'line': None, 'title': 'Unhandled database error and missing input validation', 'description': 'The callback ignores the err parameter, which causes a 200 response with undefined/empty data on query failure. Validate that req.query.id is present and valid (returning 400 if not), and check if (err) inside the callback to return an appropriate 500 error response.'}, {'serv

In [12]:
from agents.code_reviewer import ReviewFinding, CodeReview

print("ReviewFinding fields:")
print(ReviewFinding.model_fields.keys())

print("\nCodeReview schema:")
print(CodeReview.model_json_schema())

ReviewFinding fields:
dict_keys(['serverity', 'category', 'file', 'line', 'title', 'description'])

CodeReview schema:
{'$defs': {'ReviewFinding': {'properties': {'serverity': {'description': 'Serverity of the issue', 'enum': ['critical', 'High', 'Medium', 'Low', 'Info'], 'title': 'Serverity', 'type': 'string'}, 'category': {'description': 'Category of the issue', 'enum': ['bug', 'security', 'Performance', 'Quality'], 'title': 'Category', 'type': 'string'}, 'file': {'description': 'Name of the affected file', 'title': 'File', 'type': 'string'}, 'line': {'anyOf': [{'type': 'integer'}, {'type': 'null'}], 'default': None, 'description': 'Changed line number,or null if unknown', 'title': 'Line'}, 'title': {'description': 'Short issue title', 'title': 'Title', 'type': 'string'}, 'description': {'description': 'Suggested solution for the issue', 'title': 'Description', 'type': 'string'}}, 'required': ['serverity', 'category', 'file', 'title', 'description'], 'title': 'ReviewFinding', 'type':

In [13]:
result = review_code_diff(filename, diff)

print(type(result))
print(result)

<class 'dict'>
{'summary': 'The new /user endpoint in auth.js introduces a critical SQL injection vulnerability, lacks database error handling and input validation, and returns all columns from the users table directly.', 'findings': [{'serverity': 'critical', 'category': 'security', 'file': 'auth.js', 'line': None, 'title': 'SQL Injection vulnerability via query parameter concatenation', 'description': 'User input from req.query.id is concatenated directly into the SQL query string. Use a parameterized query instead, such as db.execute("SELECT id, name, email FROM users WHERE id = ?", [userId], ...).'}, {'serverity': 'High', 'category': 'bug', 'file': 'auth.js', 'line': None, 'title': 'Unhandled database error in query callback', 'description': 'The err parameter in the db.execute callback is ignored. Check for errors (if (err) { return res.status(500).json({ error: "Internal server error" }); }) before sending the response.'}, {'serverity': 'Medium', 'category': 'Quality', 'file': 'a

In [14]:
from agents.code_reviewer import (
    ReviewFinding,
    CodeReview,
    review_code_diff
)

print(ReviewFinding.model_fields.keys())

result = review_code_diff(filename, diff)

print(result)

dict_keys(['serverity', 'category', 'file', 'line', 'title', 'description'])


GoogleAPIError: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}

In [15]:
%reset -f

In [16]:
from agents.code_reviewer import ReviewFinding

print(ReviewFinding.model_fields.keys())

dict_keys(['serverity', 'category', 'file', 'line', 'title', 'description'])


In [17]:
import agents.code_reviewer as cr

print(cr.__file__)
print(cr.ReviewFinding.model_fields.keys())

c:\Users\amrit\OneDrive\Desktop\Course Project\PR-Sentinel\ai-engine\agents\code_reviewer.py
dict_keys(['serverity', 'category', 'file', 'line', 'title', 'description'])


In [18]:
import importlib
import agents.code_reviewer as cr

importlib.reload(cr)

print(cr.ReviewFinding.model_fields.keys())

dict_keys(['severity', 'category', 'file', 'line', 'title', 'description', 'suggested_fix'])
